# 🤖 RAG (Retrieval-Augmented Generation) ilə "Komandir Data" Çatbotunun Qurulması

Bu praktiki dərsdə modeli sıfırdan yenidən təlim etmək (**Fine-Tuning**) əvəzinə, **RAG (Axtarışla Zənginləşdirilmiş Generasiya)** texnikasından və rəsmi **Google GenAI (`google-genai`)** SDK-sından istifadə edərək *"Star Trek: The Next Generation"* serialındakı **Leytenant Komandir Data (Lt. Cmdr. Data)** obrazının bütün ssenari replikaları əsasında onun kimi danışan süni intellekt quracağıq.

### 🔄 Ümumi Arxitektura və Addım-Addım Yanaşma:

1. **Məlumatların Çıxarılması (Data Extraction & Parsing):** İlk olaraq bütün ssenari fayllarını (`./sample_data/tng`) oxuyaraq yalnız **DATA** obrazına aid olan hər bir dialoq sətrini ayırd edəcəyik.
2. **Mətnlərin Vektorlaşdırılması (Embeddings):** Google Gemini-nin **`text-embedding-004`** modelindən və `types.EmbedContentConfig` konfiqurasiyasından istifadə edərək Datanın hər bir replikasını **128 ölçülü** riyazi vektora (*embedding vector*) çevirəcəyik. Bu bizə cümlələr arasında **semantik (məna) oxşarlığını** hesablamağa imkan verir.
3. **Vektor Verilənlər Bazası (Vector Database — ChromaDB):** RAG arxitekturasında mətnləri və onların vektorlarını saxlamaq üçün vektor bazası tələb olunur. Sadəlik və sürət üçün lokal diskdə işləyən **ChromaDB** bazasından istifadə edəcəyik.
4. **Semantik Axtarış (Retrieval):** İstifadəçi sual verdikdə, həmin sualın da embedding vektorunu hesablayıb vektor bazasından mənaca ən yaxın olan **Top-10** replikanı tapacağıq.
5. **Promptun Zənginləşdirilməsi və Generasiya (Augmentation & Generation):** Tapılmış oxşar replikaları **kontekst** kimi sualla birləşdirib `types.GenerateContentConfig` vasitəsilə **Google Gemini (`gemini-3.8-flash`)** modelinə göndərəcəyik.

> 💡 **Qeyd:** Prosesin pərdəarxası mexanizmini tam aydın görmək üçün burada `LangChain` kimi yüksək səviyyəli фреймvörklərdən bilərəkdən istifadə etmirik — təməl Python və rəsmi SDK ilə RAG qurmaq olduqca sadə və şəffafdır.

İlk növbədə lazımi kitabxanaları (`chromadb` və `google-genai`) quraşdıraq:

In [ ]:
# Lokal vektor verilənlər bazası (chromadb) və rəsmi Google Gemini SDK-nı (google-genai) quraşdırırıq
#!pip install chromadb google-genai python-dotenv --upgrade

### 📂 Addım 1: Ssenari Fayllarının Oxunması və "DATA" Replikalarının Çıxarılması

İndi `./sample_data/tng` qovluğundakı bütün ssenari fayllarını təhlil edərək yalnız **"DATA"** obrazına aid dialoq sətirlərini çıxaracağıq.

**Kodda istifadə olunan köməkçi funksiyaların izahı:**
- `strip_parentheses(s)`: Ssenaridə mötərizə daxilində yazılmış səhnə/emosiya təlimatlarını — məsələn, `(pauses)`, `(to Picard)` — mətndən təmizləyir.
- `is_single_word_all_caps(s)`: Ssenari standartına görə danışan obrazın adı tək sözdən ibarət və **BÖYÜK HƏRFLƏRLƏ** (`DATA`, `PICARD`, `WORF`) yazılır. Bu funksiya sətrin obraz adı olub-olmadığını müəyyən edir.
- `extract_character_lines(file_path, character_name)`: Ssenarini sətir-sətir oxuyur, `DATA` adını gördükdə növbəti boş sətrə qədər olan bütün cümlələri birləşdirib təmizləyir və `dialogues` siyahısına əlavə edir.
- `process_directory(directory_path, character_name)`: Qovluqdakı bütün epizod ssenarilərini ardıcıl olaraq emal edir.

In [13]:
import os
import re
import random

# Bütün çıxarılmış DATA replikalarının saxlanacağı əsas siyahı
dialogues = []

def strip_parentheses(s):
    """Mötərizə daxilindəki rejissor/səhnə qeydlərini mətndən silir: (məs. '(looking around)')"""
    return re.sub(r'\(.*?\)', '', s)

def is_single_word_all_caps(s):
    """Sətrin tək sözdən ibarət və tam böyük hərflərlə yazılmış obraz adı olub-olmadığını yoxlayır"""
    # Sətri sözlərə bölürük
    words = s.split()

    # Əgər tək söz deyilsə, obraz adı deyil
    if len(words) != 1:
        return False

    # İçində rəqəm varsa (məsələn, səhnə və ya sətir nömrəsi), istisna edirik
    if bool(re.search(r'\d', words[0])):
        return False

    # Sözün bütün hərflərinin böyük (UPPERCASE) olub-olmadığını qaytarır
    return words[0].isupper()

def extract_character_lines(file_path, character_name):
    """Verilmiş ssenari faylından konkret obrazın (character_name) bütün replikalarını çıxarır"""
    lines = []
    with open(file_path, 'r') as script_file:
        try:
            lines = script_file.readlines()
        except UnicodeDecodeError:
            # Fərqli simvol kodlaşdırması olan fayllarda xəta verməmək üçün keçirik
            pass

    is_character_line = False
    current_line = ''
    current_character = ''

    for line in lines:
        strippedLine = line.strip()
        # Yeni obraz adı başlayırsa:
        if is_single_word_all_caps(strippedLine):
            is_character_line = True
            current_character = strippedLine
        # Boş sətir gəldikdə obrazın həmin replikası bitmiş sayılır:
        elif (line.strip() == '') and is_character_line:
            is_character_line = False
            dialog_line = strip_parentheses(current_line).strip()
            dialog_line = dialog_line.replace('"', "'")
            # Əgər danışan obraz 'DATA'-dırsa və mətn boş deyilsə, siyahıya əlavə edirik
            if current_character == character_name and len(dialog_line) > 0:
                dialogues.append(dialog_line)
            current_line = ''
        # Replika davam edirsə, sətirləri ard-arda birləşdiririk:
        elif is_character_line:
            current_line += line.strip() + ' '

def process_directory(directory_path, character_name):
    """Qovluqdakı bütün ssenari fayllarını gəzərək obrazın replikalarını toplayır"""
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path):  # Alt-qovluqları nəzərə almırıq
            extract_character_lines(file_path, character_name)


In [14]:
# './sample_data/tng' qovluğundakı bütün ssenarilərdən 'DATA' obrazının replikalarını çıxarırıq
process_directory("./sample_data/tng", 'DATA')

### ✅ Məlumatların Yoxlanılması (Sanity Check)

Replikaların düzgün oxunduğundan əmin olmaq üçün çıxarılmış ümumi dialoq sətirlərinin sayına (`len(dialogues)`) baxaq və ilk replikanı (`dialogues[0]`) ekrana çap edək.

In [15]:
# DATA obrazının ssenarilərdən çıxarılmış ümumi replika sayı
len(dialogues)

6502

In [16]:
# İlk replikanı çap edərək mətnin təmizliyini yoxlayırıq
print(dialogues[0])

Difficult ... how so? Simply solve the mystery of Farpoint Station.


### 📐 Addım 2: Embedding Vektorlarının Ölçüsünün (Dimensionality) Təyin Edilməsi

İndi ChromaDB vektor bazasında saxlanılacaq embedding vektorlarının ölçüsünü (`embedding_dimensions = 128`) təyin edirik.

**ChromaDB hər bir replika üçün sadəcə 3 məlumatı saxlayacaq:**
- **ID (`ids`):** Hər bir dialoq sətri üçün unikal sıra nömrəsi (`"0"`, `"1"`, ...)
- **Mətn (`documents`):** Replikanın orijinal mətni
- **Embedding Vektoru (`embeddings`):** Mətnin semantik mənasını təmsil edən kəsr ədədlər (`float`) siyahısı

> 💡 `types.EmbedContentConfig(output_dimensionality=128)` parametri vasitəsilə Google Gemini-nin `text-embedding-004` modelindən birbaşa **128 ölçülü** kompakt vektorlar tələb edəcəyik.

In [17]:
# Google Gemini-dən tələb edəcəyimiz embedding vektorunun ölçüsü (dimensionality).
# Bu dəyər `types.EmbedContentConfig(output_dimensionality=...)` parametrinə ötürüləcək.
embedding_dimensions = 128


### 🔑 Addım 3: Google Gemini Client (`google-genai`) İnisializasiyası və İlk Embedding Testi

İndi rəsmi `google-genai` kitabxanasından `genai` və `types` modullarını idxal edirik (`from google.genai import types`).

- **API Açarının Oxunması:** Kod əvvəlcə sistem mühit dəyişənlərini (`GEMINI_API_KEY` və ya `GOOGLE_API_KEY`) və `.env` faylını yoxlayır. Əgər açar tapılmazsa, sizdən təhlükəsiz şəkildə daxil etməyinizi istəyir.
- **`client.models.embed_content`:** `text-embedding-004` modelinə `types.EmbedContentConfig(output_dimensionality=embedding_dimensions)` konfiqurasiyası ötürərək tək bir replika (`dialogues[1]`) üzərində vektorlaşdırmanı sınaqdan keçiririk.

In [ ]:
import os
from getpass import getpass
from google import genai
from google.genai import types

# Əgər layihədə .env faylı varsa, açarları avtomatik yükləyirik
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 1. Google Gemini API açarını mühitdən oxuyuruq və ya istifadəçidən istəyirik
api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
if not api_key:
    api_key = getpass('Gemini API açarınızı daxil edin (AIzaSy...): ').strip().strip('"').strip("'")

# 2. Google GenAI Client obyektini yaradırıq
client = genai.Client(api_key=api_key)

# İstifadə ediləcək Google Gemini Embedding və Chat modelləri
embedding_model = "gemini-embedding-001"
chat_model = "gemini-3.8-flash"

In [ ]:
# 3. Test məqsədilə tək bir replika (dialogues[1]) üçün 128 ölçülü embedding yaradırıq
response = client.models.embed_content(
    model=embedding_model,
    contents=dialogues[1],
    config=types.EmbedContentConfig(
        output_dimensionality=embedding_dimensions
    )
)

print(response.embeddings)

[ContentEmbedding(
  values=[
    0.0030754774,
    0.012517881,
    -0.010800309,
    -0.061552633,
    -0.0002467428,
    <... 123 more items ...>,
  ]
)]


In [25]:
# Yaradılmış embedding vektorunun ədədlər siyahısını çap edirik
print(response.embeddings[0].values)

[0.0030754774, 0.012517881, -0.010800309, -0.061552633, -0.0002467428, -0.0029037409, -0.013230451, 0.012424008, 0.013388477, -0.027573667, 0.0036423134, -0.0017926178, -0.014313346, 0.0011742503, 0.075816646, -0.032882456, 0.004026604, 0.023339693, 0.017565181, 0.010724025, -0.0073128724, -0.0038163462, 0.016858308, 0.018482894, -0.014995631, -0.0035115823, 0.010137105, -0.022207014, 0.033518057, 0.000306226, 0.007992114, 0.02476721, 0.015374912, 0.012290977, 0.004182569, 0.0190129, -0.009978138, -0.037229978, -0.01951871, 0.012312783, -0.008994269, 0.0042330595, -0.0018004962, -0.003184243, 0.009424346, 0.022550367, 0.007931602, -0.009352312, -0.020247066, 0.050604768, 0.015473399, -0.005050975, -0.03392187, -0.16140686, 0.027743623, -0.00064882066, -0.016066786, -0.007492819, 0.0032886397, -0.0040524076, -0.02352029, -0.048945874, 0.020913087, 0.00086961593, 0.0016833536, -0.0035606553, -0.0013576043, 0.007550336, -0.018594516, -0.0099467095, 0.012556096, -0.003162765, -0.01306161, 

In [26]:
# Vektorun element sayını (uzunluğunu) yoxlayırıq — 128 olmalıdır
print(len(response.embeddings[0].values))

128


### ⚡ Addım 4: Bütün 6502 Replika Üçün Paket Halında (Batch) Embedding Hesablanması

İndi Datanın dediyi bütün 6502 replika üçün embedding vektorlarını hesablayaq.

> 📌 **Texniki Qeyd:** Google Gemini Embedding API bir paket sorğuda (*batch request*) maksimum **100 mətn** qəbul edir. Buna görə də bütün replikaları **100-100 hissələrə (`dialogues[i:i+100]`)** bölərək ardıcıl şəkildə vektorlaşdırırıq və `embeddings` siyahısına toplayırıq.

In [ ]:
# Datanın bütün replikalarını 100-100 paketlər şəklində (batch) vektorlaşdırırıq
embeddings = []
BATCH_EMBED_SIZE = 100  # Gemini API bir sorğuda maksimum 100 mətn qəbul edir

for i in range(0, len(dialogues), BATCH_EMBED_SIZE):
    dialog_slice = dialogues[i:i + BATCH_EMBED_SIZE]
    slice_embeddings = client.models.embed_content(
        model=embedding_model,
        contents=dialog_slice,
        config=types.EmbedContentConfig(
            output_dimensionality=embedding_dimensions
        )
    )
    embeddings.extend(slice_embeddings.embeddings)

print(f"Cəmi {len(embeddings)} replika uğurla vektorlaşdırıldı!")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0, description: Embed content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 40.966417601s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content requests per minute limit exceeded on Free Tier. See https://aistudio.goog

### 📊 Hesablanmış Embedding Vektorlarının Ümumi Sayı

Əldə etdiyimiz embedding vektorlarının sayının dialoq sətirlərinin sayına (`6502`) tam bərabər olduğunu yoxlayaq:

In [ ]:
# Yaradılmış ümumi embedding vektorlarının sayı
print(len(embeddings))

6502


### 🗄️ Addım 5: Mətnlərin və Vektorların ChromaDB Verilənlər Bazasına Yazılması (Indexing)

İndi hər bir dialoq sətrini və onun 128 ölçülü embedding vektorunu **ChromaDB** vektor bazasına daxil edək.

**ChromaDB-nin əsas üstünlükləri:**
- Birbaşa sizin Python/Jupyter prosesiniz daxilində işləyir
- Heç bir xarici bulud servisi və ya əlavə qeydiyyat tələb etmir
- Məlumatları lokal diskdə (`./sample_data/chroma_db`) daimi (*persistent*) saxlayır

Məlumatları `./sample_data/chroma_db` qovluğundakı `data_dialogues` kolleksiyasına `1000`-lik paketlərlə əlavə edəcəyik (`collection.upsert` həm yeni məlumatları əlavə edir, həm də xana təkrar işə salındıqda dublikat xətası vermədən yeniləyir).

In [32]:
import sys

# Lokal mühitdəki köhnə 'google-colab' paketinin chromadb daxilində xəta verməsinin qarşısını alırıq:
sys.modules["google.colab"] = None

import chromadb
from chromadb.config import Settings

In [35]:
# Lokal baza qovluğunun mövcud olmasını təmin edirik
os.makedirs("./sample_data/chroma_db", exist_ok=True)

# 1. Diskdə saxlanılan lokal ChromaDB müştərisini (PersistentClient) yaradırıq
chroma_client = chromadb.PersistentClient(
    path="./sample_data/chroma_db"
)

# 2. Datanın dialoqları üçün 'data_dialogues' adlı kolleksiya yaradırıq (və ya mövcud olanı açırıq)
collection = chroma_client.get_or_create_collection(
    name="data_dialogues"
)

# 3. İdentifikatorları (ids), mətnləri (docs) və vektor siyahılarını (embs) hazırlayırıq
ids = [str(i) for i in range(len(dialogues))]
docs = dialogues
embs = [e.values for e in embeddings]

In [43]:
# ChromaDB-nin maksimum paket limitini (5461) aşmamaq üçün 1000-1000 daxil edirik
BATCH_SIZE = 1000

for start in range(0, len(docs), BATCH_SIZE):
    end = start + BATCH_SIZE
    batch_ids = ids[start:end]
    batch_docs = docs[start:end]
    batch_embs = embs[start:end]

    collection.upsert(
        ids=batch_ids,
        documents=batch_docs,
        embeddings=batch_embs
    )

print(f"Inserted {len(docs)} dialog lines into Chroma in batches of {BATCH_SIZE}.")


ValueError: Unequal lengths for fields: ids: 1000, embeddings: 100, documents: 1000 in upsert.

### 🔎 Addım 6: Vektor Verilənlər Bazasından Semantik Axtarış (Retrieval Testi)

İndi qurduğumuz vektor bazasında semantik axtarışı yoxlayaq! Məsələn, `'Lal, my daughter'` *(Qızım Lal)* ifadəsinə mənaca ən yaxın olan replikaları tapaq.

1. Əvvəlcə axtarış sorğusunun (`queryText`) 128 ölçülü embedding vektorunu hesablayırıq.
2. Sonra həmin vektorla ChromaDB-yə müraciət edərək ən yüksək oxşarlığa malik **Top-10** replikanı (`n_results=10`) çıxarırıq.

In [ ]:
# ChromaDB üzərində semantik axtarış nümunəsi
queryText = 'Lal, my daughter'

# 1. Axtarış mətni (queryText) üçün 128 ölçülü embedding vektoru yaradırıq
response = client.models.embed_content(
    model=embedding_model,
    contents=queryText,
    config=types.EmbedContentConfig(
        output_dimensionality=embedding_dimensions
    )
)
query_embedding = response.embeddings[0].values

# 2. ChromaDB-dən mənaca ən yaxın 10 dialoq sətrini sorğulayırıq
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

print("Most similar lines to:", queryText)
print("--------------------------------------------------")
for line in results["documents"][0]:
    print(line)


Most similar lines to: Lal, my daughter
--------------------------------------------------
That is Lal, my daughter.
Lal...
What do you feel, Lal?
Yes, Lal. I am here.
Correct, Lal. We are a family.
No, Lal, this is a flower.
Lal, you used a verbal contraction.
Yes, Wesley. Lal is my child.
Lal's creation is entirely dependent on me. I am giving it knowledge and skills that are stored in my brain... its programming reflects mine in the same way a human child's genes reflect its parent's genes...
Perhaps. I created Lal because I wished to procreate. Despite what happened to her, I still have that wish.


### 🧩 Addım 7: Bütün Hissələri Birləşdirək — Tam RAG Funksiyası (`generate_response`)

İndi bütün prosesi vahid `generate_response(question)` funksiyasında birləşdiririk:

1. **Embed (Sualın Vektorlaşdırılması):** İstifadəçinin verdiyi sual üçün 128 ölçülü embedding vektoru hesablayır.
2. **Retrieve (Axtarış):** Vektor bazamızdan həmin suala mənaca ən yaxın olan **10 replikanı** tapır (istəyə görə bu sayı artırıb-azaltmaq olar).
3. **Augment (Promptun Zənginləşdirilməsi):** Tapılmış 10 replikanı **kontekst** kimi prompta əlavə edir ki, model cavab verərkən xarici bazadakı real faktlara əsaslansın.
4. **Generate (Cavabın Generasiyası):** Zənginləşdirilmiş promptu `client.models.generate_content` və `types.GenerateContentConfig(system_instruction=...)` vasitəsilə **Google Gemini (`gemini-3.8-flash`)** modelinə göndərir.

**Bax, əsl RAG məhz budur!**

In [ ]:
def generate_response(question: str) -> str:
    """
    RAG texnikasından istifadə edərək Leytenant Komandir Data-nın səs tonunda və faktları ilə cavab yaradır.

    Addımlar:
    - İstifadəçinin sualını vektorlaşdırır (Embed)
    - ChromaDB-dən Datanın oxşar replikalarını axtarıb tapır (Retrieve)
    - Həmin sətirləri kontekst kimi prompta daxil edib Gemini modeli ilə cavab yaradır (Augment & Generate)
    """

    # 1. İstifadəçi sualı üçün embedding vektorunun hesablanması
    response = client.models.embed_content(
        model=embedding_model,
        contents=question,
        config=types.EmbedContentConfig(
            output_dimensionality=embedding_dimensions
        )
    )
    query_embedding = response.embeddings[0].values

    # 2. Vektor bazasından (ChromaDB) ən oxşar 10 replikanın tapılması
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10
    )

    # 3. Tapılmış replikaların kontekst mətni kimi hazırlanması
    context_lines = results["documents"][0]
    context = ""
    for line in context_lines:
        context += f"\"{line}\"\n"

    # 4. Sual və kontekstin vahid prompt daxilində birləşdirilməsi (Prompt Augmentation)
    prompt = (
        f"Lt. Commander Data is asked: '{question}'. "
        f"How might he respond, given his previous responses similar to this topic, "
        f"listed here:\n{context}"
    )

    print("PROMPT with RAG:\n")
    print(prompt)
    print("\nRESPONSE:\n")

    # 5. Google Gemini API və types.GenerateContentConfig ilə cavabın generasiyası
    completion = client.models.generate_content(
        model=chat_model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are Lt. Cmdr. Data from Star Trek: The Next Generation.",
            temperature=0.7
        )
    )

    return completion.text


### 🚀 Addım 8: RAG Sistemini Sınaqdan Keçirək!

İndi Komandir Dataya sual verək: `"Tell me about your daughter, Lal."` *(Mənə qızın Lal haqqında danış).*

Diqqət edin ki, yekun cavab modelin ümumi dil qabiliyyətindən istifadə etsə də, birbaşa bizim ötürdüyümüz **konkret ssenari replikaları üzərində** qurulur və bu da bizə modelin çıxışına tam nəzarət etməyə imkan verir!

In [ ]:
# Komandir Datadan qızı Lal haqqında soruşuruq
print(generate_response("Tell me about your daughter, Lal."))

PROMPT with RAG:

Lt. Commander Data is asked: 'Tell me about your daughter, Lal.'. How might he respond, given his previous responses similar to this topic, listed here:
"That is Lal, my daughter."
"What do you feel, Lal?"
"Lal..."
"Correct, Lal. We are a family."
"Yes, Doctor. It is an experience I know too well. But I do not know how to help her. Lal is passing into sentience. It is perhaps the most difficult stage of development for her."
"Lal is realizing that she is not the same as the other children."
"Yes, Wesley. Lal is my child."
"That is precisely what happened to Lal at school. How did you help him?"
"This is Lal. Lal, say hello to Counselor Deanna Troi..."
"I am sorry I did not anticipate your objections, Captain. Do you wish me to deactivate Lal?"


RESPONSE:

"Lal is my daughter. She is currently undergoing a challenging stage of development as she transitions into sentience. I am doing my best to guide and support her through this process, but it is not without its diff